# M4 §3 — Elo Backtest Report

Walk-forward Elo variants vs the constant-0.5 baseline, split at 2025-08-01.
**Leakage law:** ratings use the past only; predictions use PRE-match ratings.
Engine: `src/cs2analytics/features/elo.py`. Harness: `src/cs2analytics/models/backtest.py`.

In [1]:
from pathlib import Path

import pandas as pd

from cs2analytics.models.backtest import build_backtest_table

REPO = Path.cwd()
series = pd.read_csv(REPO / "outputs" / "series_clean.csv")
RAW = REPO / "data" / "raw" / "cs2_all_tiers_games.csv"
games = pd.read_csv(RAW, low_memory=False)
d0, d1 = series["datetime"].min(), series["datetime"].max()
n = len(series)
print(f"series rows: {n} | date range: {d0} -> {d1}")

series rows: 9920 | date range: 2023-01-10T09:30:00Z -> 2026-06-28T20:00:00Z


In [2]:
table = build_backtest_table(series, games=games)
pd.set_option("display.width", 200)
table

,model,split,logloss,brier,acc,n
0,constant_0.5,test,0.693147,0.250000,0.558530,3511
1,elo_k16,test,0.660476,0.234386,0.603247,3511
2,elo_k32,test,0.656187,0.232322,0.606665,3511
3,elo_k32,train,0.667096,0.237408,0.588079,6409
4,elo_k64,test,0.664971,0.234978,0.612076,3511
5,elo_k8,test,0.669111,0.238323,0.594702,3511
6,elo_mapspecific_k32,test,0.680712,0.243777,0.556291,3171


## Reading the table

- `constant_0.5` = always predict 0.5: logloss 0.6931 (= ln 2), brier 0.25 — the floor any
  real model must beat.
- `elo_k32` beats it clearly on test and the K-sweep peaks at K=32: slower ratings (K=8)
  adapt too slowly, faster (K=64) overreact.
- `elo_mapspecific_k32` is WORSE than global Elo: per-(team, map) ratings split the
  evidence; most (team, map) pairs are too sparse. Map info belongs in features (M7),
  not in a sparse rating.
- Train vs test logloss for k32: no degradation — walk-forward ratings never "age out",
  so the split boundary is not a distribution shock here.

In [3]:
out = REPO / "outputs" / "m4_backtest_results.csv"
table.to_csv(out, index=False)
print(f"wrote {out.relative_to(REPO)}")

wrote outputs\m4_backtest_results.csv
